# 03 – Sentiment Analysis
In this part, I create text-based features like word counts, caps usage, exclamation marks, and simple sentiment scoring.


In [ ]:
USE DATABASE AMAZON_REVIEWS_DB;
USE SCHEMA ANALYTICS;
USE WAREHOUSE COMPUTE_WH;


In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np

session = get_active_session()
session


In [ ]:
reviews = session.table("REVIEWS_CLEAN").to_pandas()
reviews.head()


### Creating text features
Extracting things like text length, word count, positive/negative keywords, and a basic sentiment score.


In [ ]:
import re

reviews["TEXT_LENGTH"] = reviews["REVIEW_TEXT"].str.len()
reviews["WORD_COUNT"] = reviews["REVIEW_TEXT"].str.split().str.len()

reviews["CAPS_RATIO"] = reviews["REVIEW_TEXT"].apply(
    lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
)

reviews["EXCLAMATION_COUNT"] = reviews["REVIEW_TEXT"].str.count("!")

reviews["IS_LONG_REVIEW"] = (reviews["WORD_COUNT"] > 50).astype(int)

positive_words = ["good", "great", "amazing", "love", "excellent"]
negative_words = ["bad", "terrible", "poor", "hate", "awful"]

def count_words(text, word_list):
    text_lower = text.lower()
    return sum(text_lower.count(w) for w in word_list)

reviews["POSITIVE_WORDS"] = reviews["REVIEW_TEXT"].apply(lambda x: count_words(x, positive_words))
reviews["NEGATIVE_WORDS"] = reviews["REVIEW_TEXT"].apply(lambda x: count_words(x, negative_words))

reviews["SENTIMENT_SCORE"] = reviews["POSITIVE_WORDS"] - reviews["NEGATIVE_WORDS"]

reviews.head()


In [ ]:
feature_df = reviews[[
    "REVIEW_ID",
    "PRODUCT_ID",
    "USER_ID",
    "RATING",
    "TEXT_LENGTH",
    "WORD_COUNT",
    "CAPS_RATIO",
    "EXCLAMATION_COUNT",
    "IS_LONG_REVIEW",
    "POSITIVE_WORDS",
    "NEGATIVE_WORDS",
    "SENTIMENT_SCORE"
]]

feature_df.head()


### Saving the engineered dataset
Writing the enhanced dataset back into Snowflake for dashboarding later.


In [ ]:
features_sf = session.create_dataframe(feature_df)

features_sf.write.save_as_table(
    "FEATURE_ENGINEERED_REVIEWS",
    mode="overwrite"
)


In [ ]:
SELECT * FROM FEATURE_ENGINEERED_REVIEWS LIMIT 20;


In [ ]:
print("Engineered feature table created:")
print("Rows:", len(feature_df))
print("Columns:", list(feature_df.columns))
